# CNN architectures: ResNet on CIFAR-10 walkthrough

This notebook is a guided companion to the chapter scripts, not a replacement for them. Use it to inspect architecture choices interactively: variant, block type, stem, stage shapes, classifier head, parameter count, and the quick synthetic-data verification path. Keep the `.py` scripts as the source of truth for repeatable runs and saved evidence artifacts.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

In [ ]:
from pathlib import Path
import subprocess
import sys
from types import SimpleNamespace


def find_code_dir():
    script_name = "resnet50_cifar10_pytorch.py"
    chapter_name = "chapter_cnn_architectures"
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError(
        "Run this notebook from the repository root or chapter_cnn_architectures."
    )


CODE_DIR = find_code_dir()
REPO_ROOT = CODE_DIR.parent
sys.path.insert(0, str(CODE_DIR))

import resnet50_cifar10_pytorch as resnet_script

print(f"Using companion code: {CODE_DIR}")
print("Available variants:", ", ".join(resnet_script.MODEL_CONFIGS))

## Build architecture objects

The PyTorch script exposes the model builder used by the smoke checks and verification run. The notebook calls that builder directly so the architecture being inspected is the same one used by the command-line script.

In [ ]:
try:
    import torch
    from torch import nn
except Exception as exc:
    torch = None
    nn = None
    print("PyTorch is not available in this kernel.")
    print("Install the PyTorch companion-code stack or switch to that environment.")
    print(exc)

try:
    from IPython.display import Image, Markdown, display
except Exception:
    Image = None
    Markdown = None
    display = None


def args_for(variant="resnet18", stem="cifar"):
    return SimpleNamespace(
        model_variant=variant,
        stem=stem,
        normalization="batchnorm",
        dropout_rate=0.0,
    )


def markdown_table(headers, rows):
    header = "| " + " | ".join(headers) + " |"
    divider = "| " + " | ".join("---" for _ in headers) + " |"
    body = ["| " + " | ".join(str(value) for value in row) + " |" for row in rows]
    return "\n".join([header, divider, *body])


def show_markdown(text):
    if Markdown is not None and display is not None:
        display(Markdown(text))
    else:
        print(text)


def format_shape(shape):
    return " x ".join(str(part) for part in shape)

The next cell builds each ResNet variant and compares the parameter counts. If PyTorch is unavailable, it prints a dependency reminder instead of stopping the notebook.

In [ ]:
if torch is None:
    print("Install PyTorch to build and compare the models.")
else:
    rows = []
    for variant in ("resnet18", "resnet34", "resnet50", "tiny-resnet"):
        block_type, block_counts = resnet_script.MODEL_CONFIGS[variant]
        model = resnet_script.build_model(nn, args_for(variant=variant, stem="cifar"))
        rows.append(
            [
                variant,
                block_type,
                "-".join(str(count) for count in block_counts),
                f"{resnet_script.parameter_count(model):,}",
            ]
        )

    show_markdown(
        markdown_table(
            ["Variant", "Block type", "Stage blocks", "Parameters"],
            rows,
        )
    )

## Compare the CIFAR stem with the ImageNet stem

The chapter stresses that a model name is not enough. On CIFAR-10, the first downsampling decision changes the whole shape trace. A CIFAR-style stem keeps the early feature map at 32 x 32; an ImageNet-style stem shrinks it immediately.

In [ ]:
def shape_trace(model):
    x = torch.zeros(1, 3, 32, 32)
    steps = [("input", tuple(x.shape))]
    modules = [
        ("stem", model.stem),
        ("layer1", model.layer1),
        ("layer2", model.layer2),
        ("layer3", model.layer3),
        ("layer4", model.layer4),
        ("global average pool", model.avgpool),
        ("flatten", model.flatten),
        ("dropout", model.dropout),
        ("classifier", model.fc),
    ]
    model.eval()
    with torch.no_grad():
        for name, module in modules:
            x = module(x)
            steps.append((name, tuple(x.shape)))
    return steps


if torch is None:
    print("Install PyTorch to inspect shape traces.")
else:
    for stem in ("cifar", "imagenet"):
        model = resnet_script.build_model(nn, args_for(variant="resnet18", stem=stem))
        rows = [[name, format_shape(shape)] for name, shape in shape_trace(model)]
        show_markdown(f"### ResNet18 with `{stem}` stem\n" + markdown_table(["Step", "Shape"], rows))

## Run the quick verification path

This is the same synthetic-data smoke path exposed by the companion script. It is useful for checking the local environment and the training loop without downloading CIFAR-10 or spending a long time training.

In [ ]:
cmd = [
    sys.executable,
    str(CODE_DIR / "resnet50_cifar10_pytorch.py"),
    "--quick",
    "--synthetic-data",
    "--batch-size",
    "16",
    "--allow-missing-deps",
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=CODE_DIR, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"quick verification failed with exit code {result.returncode}")

## Preview companion-code evidence artifacts

The companion code includes bounded verification artifacts under `reference_artifacts/resnet50-cifar10-pytorch/`. This cell displays the training-curve artifact when it is present.

In [ ]:
figure_path = CODE_DIR / "reference_artifacts" / "resnet50-cifar10-pytorch" / "training-curves.png"
if figure_path.exists() and Image is not None and display is not None:
    display(Image(filename=str(figure_path)))
elif figure_path.exists():
    print(figure_path)
else:
    print("No companion-code training-curve artifact found at", figure_path)

## Fill the architecture report template

The homework asks for an architecture record, not just a training score. Use this final cell as a starting point, then replace the rationale and cost notes with your own evidence.

In [ ]:
if torch is None:
    print("Install PyTorch to fill this template automatically.")
else:
    selected_variant = "resnet18"
    selected_stem = "cifar"
    block_type, block_counts = resnet_script.MODEL_CONFIGS[selected_variant]
    model = resnet_script.build_model(nn, args_for(variant=selected_variant, stem=selected_stem))
    trace = shape_trace(model)
    rows = [
        ["Architecture ID", f"{selected_variant}-{selected_stem}"],
        ["Variant and block type", f"{selected_variant}, {block_type} blocks"],
        ["Input and stem", "32 x 32 RGB input, 3 x 3 stride-1 CIFAR stem, no initial max pool"],
        ["Stages", "-".join(str(count) for count in block_counts)],
        ["Classifier head", "global average pooling plus 10 logits"],
        ["Parameter count", f"{resnet_script.parameter_count(model):,}"],
        ["Shape trace", " -> ".join(f"{name}: {format_shape(shape)}" for name, shape in trace)],
        ["Architecture rationale", "Keeps early CIFAR-10 spatial detail while using a student-scale ResNet."],
    ]
    show_markdown(markdown_table(["Report field", "Recorded value"], rows))

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.